In [16]:
#charger les bibliothèques nécessaires et la clé d'API
from dotenv import load_dotenv

#charger la clé d'API à partir du fichier .env
load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FastEmbedEmbeddings

print("Importation des bibliothèques et chargement de la clé d'API réussis.")


Importation des bibliothèques et chargement de la clé d'API réussis.


In [11]:
#charger les documents PDF à partir du répertoire spécifié
#chaque page est un document séparé, ce qui permet une meilleure granularité pour les recherches ultérieures
#documents -> c'est le nombre de pages de l'ensemble des PDF chargés, pas le nombre de fichiers PDF

loader = DirectoryLoader('documents/', glob='*.pdf', loader_cls=PyPDFLoader)
documents = loader.load()

print(f"{len(documents)} documents PDF chargés avec succès.")

91 documents PDF chargés avec succès.


In [12]:
#decouper les documents en morceaux de texte plus petits pour une meilleure gestion et recherche
text_splitter = RecursiveCharacterTextSplitter(chunk_size=30000, chunk_overlap=300)
texts = text_splitter.split_documents(documents)

print(f"{len(texts)} morceaux de texte créés à partir des documents PDF.")

91 morceaux de texte créés à partir des documents PDF.


In [13]:
# Embeddings LOCAUX : gratuits, sans quota, sans API
embeddings = FastEmbedEmbeddings()

# Plus besoin de lots ni de pauses ! On fait tout d'un coup, en local
vectorstore = FAISS.from_documents(texts, embeddings)

vectorstore.save_local("faiss_index")
print("Base FAISS créée et sauvegardée localement avec succès.")


Base FAISS créée et sauvegardée localement avec succès.


In [30]:
#poser des questions à l'agent RAG


llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash-lite", temperature=0.2)

#le chercheur va trouver les 4 morceaux de texte les plus pertinents pour la question posée en sens (on peut ajuster ce 
# nombre avec le paramètre "k" en fonction de la question et de la taille des morceaux de texte)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})


#fonction pour repondre à une question en utilisant le modèle de langage et les morceaux de texte pertinents
def repondre_a_question(question):
    #récupérer les morceaux de texte les plus pertinents pour la question posée dans les documents PDF
    morceaux_pertinents = retriever.invoke(question)

    #mettre les morceaux de texte pertinents dans le contexte pour que le modèle 
    #de langage puisse les utiliser pour répondre à la question
    contexte = "" 
    for passage in morceaux_pertinents:
        contexte += passage.page_content + "\n\n"

    #poser la question au modèle de langage en lui fournissant le contexte des morceaux de texte pertinents
    message = f"""Réponds à la question en utilisant SEULEMENT le contexte ci-dessous.
    Si la réponse n'y est pas, dis simplement que je n'' ai pas les compétences pour répondre à cette question.
    
    CONTEXTE:
    {contexte}
    QUESTION: {question}
    RÉPONSE:"""
        #envoyer la question et le contexte au modèle de langage pour obtenir une réponse
    reponse = llm.invoke(message)
        
        #afficher la réponse obtenue du modèle de langage
    #print("REPONSE OBTENUE DU MODÈLE DE LANGAGE :")
    print(reponse.content)
    
    #afficher les sources utilisées pour répondre à la question (les morceaux de texte pertinents)
    #print("\nSOURCES UTILISÉES :")
    #for passage in morceaux_pertinents:
        #print("-", passage.metadata.get("source", "inconnue"))  # Affiche les 200 premiers caractères de chaque passage pertinent
    

#exemple de question à poser à l'agent RAG
repondre_a_question("c'est quoi intelligence artificielle ?")
repondre_a_question("Quelles sont des applications concrètes de l'intelligence artificielle ?")
repondre_a_question("quelle est les origines ou l'histoire de l'intelligence artificielle ?")

L'intelligence artificielle (IA) est l'ensemble des systèmes informatiques capables d'effectuer des tâches typiquement associées à l'intelligence, telles que l'apprentissage, le raisonnement, la résolution de problèmes, la perception ou la prise de décision. C'est également le champ de recherche visant à développer de tels systèmes. L'IA est un domaine de l'informatique qui s'appuie sur des fondements mathématiques et des concepts issus des sciences cognitives. Elle vise à résoudre des problèmes à forte complexité logique ou algorithmique. Par extension, dans le langage courant, l'IA inclut les dispositifs imitant, simulant ou remplaçant l'homme dans certaines mises en œuvre de ses fonctions cognitives.
Les applications de l'IA couvrent de nombreux domaines, notamment les moteurs de recherche, les systèmes de recommandation, l'aide au diagnostic médical, la compréhension du langage naturel, les voitures autonomes, les chatbots, les outils de génération d'images, les outils de prise de 